# Limpieza y unificación de datos — criptomonedas

Cinco fuentes CSV independientes (BTC, DOGE, NEO, UNI, ZEN) llegan con formatos inconsistentes: distintos separadores, codificación BOM, separador de miles con punto y decimal con coma. El objetivo de este notebook es unificarlos en un único dataset limpio que sirva de input para el resto del análisis.

Resultado esperado: `datos/procesados/precios_diarios.csv` — un único CSV con 6 columnas estándar y sin filas inválidas.

In [1]:
# 01_limpieza_datos.ipynb
# Limpieza y unificación de CSV de criptomonedas
# Totalmente robusto ante inconsistencias de Excel

import pandas as pd
import os
from glob import glob

## 1. Configuración de rutas

La carpeta de crudos se lee con `glob` de forma dinámica. Añadir una nueva criptomoneda solo requiere depositar su CSV en `datos/crudos/`; el pipeline la procesa sin cambios en el código.

In [2]:
# -----------------------------
# 1. Configuración de carpetas
# -----------------------------
carpeta_crudos = "../datos/crudos/"
carpeta_procesados = "../datos/procesados/"

os.makedirs(carpeta_procesados, exist_ok=True)

archivos = glob(os.path.join(carpeta_crudos, "*.csv"))

if not archivos:
    raise FileNotFoundError(f"No se encontraron archivos CSV en {carpeta_crudos}")

lista_dfs = []

## 2. Lectura y limpieza robusta

Este bloque contiene las decisiones críticas del notebook:

**Detección automática de separador** (`sep=None, engine='python'`): los cinco CSVs no usan el mismo delimitador. Forzar `,` o `;` corrompería silenciosamente los archivos que usan el otro.

**Eliminación de BOM** (`\ufeff`): un BOM invisible en el nombre de la primera columna haría fallar cualquier `df['fecha']` o `df.rename()` posterior sin mensaje de error claro.

**Orden de conversión de precios**: primero se elimina el separador de miles (`.`), luego la coma decimal se sustituye por punto, y finalmente se convierte a float. Invertir el orden produciría errores en todos los valores con miles.

**`errors='coerce'`**: convierte valores inválidos a NaN en lugar de abortar el pipeline. Permite eliminarlos con `dropna()` y cuantificar cuántas filas se pierden sin detener el proceso.

**`drop_duplicates(subset=['cripto_id', 'fecha'])`**: se aplica al final, una vez que las fechas son tipo `datetime`. Aplicarlo antes haría comparaciones entre strings con formatos distintos.

In [ ]:
# -----------------------------
# 2. Lectura y limpieza robusta
# -----------------------------
for archivo in archivos:
    print(f"\n{'='*60}")
    print(f"Procesando archivo: {archivo}")
    print(f"{'='*60}")

    # Nombre de la cripto
    cripto_id = os.path.basename(archivo).split(".")[0].upper()

    # Leer CSV detectando separador automáticamente y quitando comillas
    df = pd.read_csv(archivo, encoding="utf-8", sep=None, engine='python', quotechar='"', skipinitialspace=True)
    print(f"  [1/6] Lectura inicial: {len(df)} filas, columnas originales: {list(df.columns)}")

    # Limpiar nombres de columnas y caracteres invisibles
    df.columns = df.columns.str.strip().str.lower().str.replace('﻿','')  # BOM
    df = df.map(lambda x: str(x).strip() if isinstance(x, str) else x)
    print(f"  [2/6] Columnas normalizadas (BOM/espacios eliminados): {list(df.columns)}")

    # Renombrar columnas a español
    df = df.rename(columns={
        "ticker": "cripto_id",
        "date": "fecha",
        "open": "apertura",
        "high": "maximo",
        "low": "minimo",
        "close": "cierre"
    })

    # Sobrescribir cripto_id por seguridad
    df["cripto_id"] = cripto_id
    print(f"  [3/6] Columnas renombradas a español. cripto_id fijado a '{cripto_id}'")

    # Verificar columnas mínimas
    columnas_esperadas = ["cripto_id","fecha","apertura","maximo","minimo","cierre"]
    filas_antes = len(df)
    df = df.dropna(subset=columnas_esperadas)  # elimina filas incompletas
    print(f"  [4/6] Filas incompletas descartadas: {filas_antes - len(df)} (quedan {len(df)})")

    # Convertir fecha a datetime
    filas_antes = len(df)
    df["fecha"] = pd.to_datetime(df["fecha"], dayfirst=True, errors='coerce')
    df = df.dropna(subset=["fecha"])  # eliminar fechas inválidas
    print(f"  [5/6] Fechas inválidas descartadas: {filas_antes - len(df)} (quedan {len(df)})")

    # Limpiar y convertir precios a float
    filas_antes = len(df)
    for col in ["apertura","maximo","minimo","cierre"]:
        df[col] = df[col].astype(str)
        df[col] = df[col].str.replace(".", "", regex=False)  # eliminar separador de miles
        df[col] = df[col].str.replace(",", ".", regex=False) # reemplazar coma decimal
        df[col] = pd.to_numeric(df[col], errors='coerce')     # convertir a float
    df = df.dropna(subset=["apertura","maximo","minimo","cierre"])  # eliminar filas inválidas
    print(f"  [6/6] Precios no numéricos descartados: {filas_antes - len(df)} (quedan {len(df)})")

    # Ordenar por fecha y eliminar duplicados
    filas_antes = len(df)
    df = df.sort_values("fecha")
    df = df.drop_duplicates(subset=["cripto_id", "fecha"])
    print(f"  Duplicados (cripto_id, fecha) eliminados: {filas_antes - len(df)}")
    print(f"  Rango de fechas final: {df['fecha'].min().date()} -> {df['fecha'].max().date()}")
    print(f"  Total final para {cripto_id}: {len(df)} filas")

    # Añadir a la lista
    lista_dfs.append(df)

## 3. Concatenación

`ignore_index=True` es obligatorio: cada DataFrame individual tiene su propio RangeIndex (0, 1, 2…). Sin él, el dataset final tendría valores de índice duplicados y cualquier `.loc[]` por índice produciría resultados incorrectos.

In [ ]:
# -----------------------------
# 3. Concatenar todos los DataFrames
# -----------------------------
print(f"Concatenando {len(lista_dfs)} DataFrames individuales:")
for d in lista_dfs:
    print(f"  {d['cripto_id'].iloc[0]}: {len(d)} filas")

df_final = pd.concat(lista_dfs, ignore_index=True)
print(f"\nDataset concatenado: {len(df_final)} filas totales, {len(df_final.columns)} columnas")

## 4. Persistencia del resultado

El separador `;` evita conflictos con valores numéricos que pudieran contener coma decimal. UTF-8 garantiza compatibilidad con pandas, Power BI y Excel sin necesidad de especificar codificación en la lectura.

In [ ]:
# -----------------------------
# 4. Guardar dataset procesado
# -----------------------------
ruta_procesado = os.path.join(carpeta_procesados, "precios_diarios.csv")
print(f"Guardando dataset unificado en: {ruta_procesado}")
df_final.to_csv(ruta_procesado, index=False, encoding="utf-8", sep=';')
print(f"Guardado completado: {os.path.getsize(ruta_procesado) / 1024:.1f} KB")

## 5. Validación del resultado

Se verifica visualmente que las fechas están bien parseadas, los precios son numéricos y la `cripto_id` se asignó correctamente. Un dataset silenciosamente corrupto aquí propagaría errores difíciles de detectar en los notebooks siguientes.

In [ ]:
# -----------------------------
# 5. Resumen final
# -----------------------------
print("Dataset unificado y guardado en:", ruta_procesado)
print("Número de registros:", len(df_final))
print("\nRegistros por criptomoneda:")
print(df_final["cripto_id"].value_counts())
print("\nTipos de dato por columna:")
print(df_final.dtypes)
print("\nValores nulos por columna (debe ser 0 en todas):")
print(df_final.isna().sum())
print("\nRango de fechas del dataset completo:", df_final["fecha"].min().date(), "->", df_final["fecha"].max().date())
print("\nPrimeros registros:\n", df_final.head())